# Lesson 51 · MFCCs on real audio — features meet the real world: a hands-on MFCC pipeline on WAV speech (librosa as an optional answer key)

**Goal**: really run a real WAV file through end to end: read the file, cut it into frames, take the STFT, extract the mel spectrum, compute MFCCs — then use four layers of plots to see vowel harmonics and consonant transients clearly.

> **Hold the no-wrapper line**: the implementation and the main path are still our own `aurora.audio`; **librosa is only an optional answer key**, and the main flow works fine without it. If loading from a path fails, first check the working directory and whether the wav file exists.

🔗 **Aurora connection**:
- `aurora.audio.io.read_wav` → `aurora.audio.stft.magnitude_spectrogram` → `aurora.audio.mel.mel_spectrogram` → `aurora.audio.mfcc.mfcc`
- the whole chain is NumPy-only, with no librosa / scipy.signal dependency


← **Previous lesson** · [L50 · The full MFCC pipeline](L50_mfcc.ipynb)

> Last lesson covered **the full MFCC pipeline**: signal → STFT → mel → log → DCT, checking the output shape at every step.
> This lesson explores **MFCCs on real audio**.

## This lesson's story: what can one "ahh——" tell a machine?

Record yourself saying "ahh" for one second and you get 16000 samples. Hidden in that pile of numbers are:
- **your pitch (F0)**: how fast your vocal folds vibrate — about 100 Hz for a low male voice, about 300 Hz for a high female voice
- **your vowel (formants)**: the resonant frequencies of your vocal tract; F1/F2 decide the difference between "a/e/i/o/u"
- **your timbre**: vocal-fold tension, how nasal you sound, and so on

MFCC is the standard tool for distilling structure out of that pile of numbers. This section runs the whole pipeline on a real WAV file (or synthetic speech) so the features really "come alive":

**Four layers of representation, from surface to depth**:

| Layer | Tool | What to look at |
|---|---|---|
| time-domain waveform | line plot | amplitude over time; vowel segments carry high energy |
| linear spectrogram | STFT → `imshow` | harmonic structure (evenly spaced bright lines) |
| mel spectrogram | `mel_spectrogram` | compressed, perceptual frequency |
| MFCC heatmap | `mfcc()` | 13 cepstral coefficients changing frame by frame |

**Typical parameters**: `sr=16000, n_fft=1024, hop=256, n_mels=40, n_mfcc=13`

This section's task: do 3 exercises and build the experimental habit of "change a parameter → observe what happens → explain why".

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import read_wav, sine, write_wav
from aurora.audio.stft import magnitude_spectrogram
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import mfcc


## A 2-minute review before we start: three pitfalls of real WAV files

- The sample rate can differ: 8k / 16k / 44.1k, and it must always travel together with the samples
- Make mono / stereo consistent first, then compute features
- Ideally normalize the amplitude to `[-1, 1]` first

Here `librosa` is only an answer key, not a production dependency; the "MFCC on LibriSpeech" item in the `ROADMAP` means hooking this pipeline up to a real dataset and a training task.
If your memory of harmonics / mel / DCT from earlier is hazy, go back to L03, L07, L46, L47 first.

## Going deeper: the 1:2 relationship between sample rate and Nyquist frequency

You may have noticed in the table in the next section: "sample rate 16000 Hz → Nyquist 8000 Hz". Why does it have to be divided by 2?

**An intuitive story**: imagine recording a 5 Hz sine wave (5 oscillations per second). If you only sample at 10 Hz (10 points per second), can you "see" the wave?

```
5 Hz sine wave:     sin, sin↓, sin↓↓, sin↓, sin, sin↑, ...
10 Hz samples:      ●     ●      ●     ●    ●    ●
(one point every 0.1 s)
```

You can! Because 10 > 5, you take at least 2 points in every cycle, which is enough to reconstruct it.

Now try sampling the same 5 Hz sine wave at 5 Hz:

```
5 Hz sine wave:     sin, sin↓, sin↓↓, sin↓, sin, sin↑, ...
5 Hz samples:       ●          ●          ●
(one point every 0.2 s, landing only on the peaks)
```

Game over! What you sampled looks like a constant (all peaks) — completely distorted. This is called **aliasing**.

**The truth of the sampling theorem**: to capture a signal of frequency $f$ without distortion, the sample rate must be $\geq 2f$. Put the other way round, a system with sample rate $sr$ can capture signals only up to frequency $sr/2$. This $sr/2$ is called the **Nyquist frequency**.

**Why a 1:2 relationship?** Because sketching out one full cycle takes **at least 2 samples** (one on the rising part, one on the falling part).

So the rule in the table can be read like this:
- sample rate 16000 Hz → can capture signals up to 8000 Hz → any component > 8000 Hz gets "folded back" and aliases
- in the STFT/MFCC, the spectrogram only shows 0 – 8000 Hz; there's no need to show 8000 – 16000 Hz (it's all mirror image there)

## 1. `read_wav()` — the WAV container and the sample rate (sr)

`read_wav(path)` returns `(samples: np.ndarray[float64], sr: int)`.
It parses the RIFF/WAVE header by hand (the `wave` standard library), supports 8/16/32-bit PCM, averages multiple channels down to mono, and normalizes the output to `[-1, 1]`.

**Sample rates differ**:

| Source | sr (Hz) | Nyquist | Typical use |
|---|---|---|---|
| telephone speech | 8 000 | 4 000 Hz | ASR baseline |
| LibriSpeech | 16 000 | 8 000 Hz | modern ASR |
| broadcast / music | 44 100 | 22 050 Hz | music AI |

At a different sample rate, the same `hop_length` and `n_fft` cover a different physical duration (`hop_length/sr` seconds), and the frequency resolution `sr/n_fft` changes too — so `sr` must be passed to downstream functions together with `samples`.

The code cell below first tries to load `test_speech.wav`; if the file doesn't exist, it uses aurora to synthesize a vowel–consonant–vowel (V-C-V) imitation, writes it to disk and reads it back, so the pipeline is the same either way.


In [ ]:
WAV_PATH = "test_speech.wav"

def make_vcv_signal(sr: int = 16000) -> np.ndarray:
    """Synthesize a V-C-V sequence:
    [0.0-0.4s] vowel /a/ — 120 Hz fundamental + harmonics (energy concentrated in the low formants)
    [0.4-0.5s] plosive consonant — a short burst of broadband white noise
    [0.5-1.0s] vowel /i/ — 150 Hz fundamental + different harmonic weights (high formants stand out)
    """
    t_a = np.arange(int(0.4 * sr)) / sr
    vowel_a = sum(
        (1.0 / k) * np.sin(2 * np.pi * 120 * k * t_a)
        for k in range(1, 9)
    )
    vowel_a /= np.max(np.abs(vowel_a)) + 1e-9

    n_burst = int(0.1 * sr)
    burst = np.random.default_rng(42).standard_normal(n_burst) * 0.3

    t_i = np.arange(int(0.5 * sr)) / sr
    harmonic_weights = [1.0, 0.4, 0.8, 0.2, 0.6, 0.1, 0.5, 0.15]
    vowel_i = sum(
        w * np.sin(2 * np.pi * 150 * k * t_i)
        for k, w in enumerate(harmonic_weights, start=1)
    )
    vowel_i /= np.max(np.abs(vowel_i)) + 1e-9

    return np.concatenate([vowel_a, burst, vowel_i])

import os
if not os.path.exists(WAV_PATH):
    print("test_speech.wav not found; synthesizing a V-C-V test signal…")
    sr_synth = 16000
    signal_synth = make_vcv_signal(sr_synth)
    write_wav(WAV_PATH, signal_synth, sr_synth)
    print(f"wrote {WAV_PATH}  ({len(signal_synth)/sr_synth:.2f}s, {sr_synth} Hz)")

samples, sr = read_wav(WAV_PATH)
duration = len(samples) / sr
print(f"loaded: {len(samples)} samples, sr={sr} Hz, duration={duration:.3f}s")
print(f"amplitude range: [{samples.min():.4f}, {samples.max():.4f}]")

assert samples.dtype == np.float64
assert -1.0 <= samples.min() and samples.max() <= 1.0
print("✅ read_wav output format is correct")


## The physics of vowels: why do harmonics fall off as 1/k?

How is your voice produced? In short: **vocal folds + vocal-tract resonance**.

1. **The vocal folds vibrate**: two bands of muscle in your throat vibrate like a guitar string, producing a **fundamental frequency (F0)**. About 100–150 Hz for men, about 200–300 Hz for women.

2. **Multiples appear**: but the vocal folds don't oscillate as an ideal sine wave. In reality their vibration is shaped more like a **periodic square wave** or sawtooth. Any periodic signal can be decomposed into the fundamental + whole-number multiples, the harmonics (2× the fundamental, 3× the fundamental, ...) — that's the heart of the **Fourier series**.

3. **Where the 1/k fall-off comes from**: for some common periodic waveforms (square waves, sawtooth waves), the harmonic amplitudes naturally fall off as $1/k$. It isn't something we chose; it's **physics**. For example:
   - fundamental (k=1) amplitude: 1.0
   - 2nd harmonic (k=2) amplitude: 1/2 = 0.5
   - 3rd harmonic (k=3) amplitude: 1/3 ≈ 0.33
   - ...

4. **Filtering by the vocal tract**: the 1/k harmonics from the vocal folds pass through the **resonators** — your mouth, nasal cavity, tongue position and so on. Different tongue positions and mouth shapes boost some harmonics and weaken others. These resonances are called **formants**.
   - vowel /a/ (as in "father"): formants usually lower in frequency
   - vowel /i/ (as in "see"): formants higher in frequency, so the higher harmonics are weighted differently

In `make_vcv_signal()`, 1/k stands for the vocal folds' 1/k fall-off; then /a/ and /i/ each get a different weight vector, imitating how vocal-tract resonance selectively amplifies different harmonics.

## 2. Optional check: comparing with librosa

Per `CLAUDE.md`: **the Audio Core does not bring in librosa**. Aurora's STFT, mel and MFCC are all hand-written from the formulas and tested against `numpy.fft` as the reference (`tests/audio/`).

For teaching or debugging, we can **optionally** bring in librosa for a one-off numerical comparison, confirming that Aurora's results agree numerically:

```
Aurora mel_spectrogram  vs  librosa.feature.melspectrogram
max relative difference < 1e-4  ✓
```

The cell below runs the comparison automatically when librosa is available; if it isn't installed, it's skipped and the main flow is unaffected. This is Aurora's design philosophy in action: **core algorithms built ourselves, black-box libraries used only for external validation**.


In [ ]:
# Optional: check against librosa (for validation only, not a dependency)
try:
    import librosa  # type: ignore
    _HAS_LIBROSA = True
except ImportError:
    _HAS_LIBROSA = False
    print("librosa not installed; skipping the comparison (the main flow is unaffected)")

if _HAS_LIBROSA:
    n_fft = 1024
    hop  = n_fft // 4

    aurora_mel = mel_spectrogram(samples, sr, n_fft=n_fft, hop_length=hop, n_mels=80)
    aurora_db  = power_to_db(aurora_mel)

    lr_mel = librosa.feature.melspectrogram(
        y=samples.astype(np.float32), sr=sr,
        n_fft=n_fft, hop_length=hop, n_mels=80,
        power=2.0, center=True,
    )
    lr_db = librosa.power_to_db(lr_mel)

    # aurora shape: (n_frames, n_mels); librosa shape: (n_mels, n_frames)
    aurora_db_T = aurora_db.T  # -> (n_mels, n_frames)
    min_frames  = min(aurora_db_T.shape[1], lr_db.shape[1])
    diff = np.abs(aurora_db_T[:, :min_frames] - lr_db[:, :min_frames])
    print(f"Mel-dB max difference: {diff.max():.4f} dB, mean difference: {diff.mean():.4f} dB")

    # Why 2 dB?
    # 2 dB ≈ a 1.26× difference in amplitude, ≈ 1.58× in power (20 dB = 10× in amplitude, so 2 dB = 10^(2/20) ≈ 1.26)
    # The threshold allows for numerical precision differences (floating-point rounding, algorithmic details) and small implementation differences between libraries
    # but is strict enough to catch algorithm bugs (a wrong window function, a badly implemented mel filter, etc.)
    assert diff.max() < 2.0, "Aurora and librosa mel differ too much; check mel.py"
    print("✅ Aurora mel_spectrogram agrees numerically with librosa (< 2 dB)")
    print("   (2 dB allows for floating-point error but is enough to catch implementation bugs)")

## 3. Visual check: four layers of representation

The four layers of visualization go from surface to depth, showing what the same stretch of speech looks like at different levels of abstraction:

| Layer | Axes | What to look at |
|---|---|---|
| **waveform** | time / amplitude | energy envelope, silences, plosive transients |
| **magnitude spectrogram** | time / linear Hz | harmonic stripes (vowels), broadband flashes (consonants) |
| **log-mel** | time / mel band | formant tracks, perceptual scale |
| **MFCC** | time / coefficient index | compact speech features, C0 ≈ energy |

**How to tell vowels and consonants apart**:
- vowel regions: several **horizontal parallel bright stripes** stacked up the spectrogram (the harmonics), spaced by the fundamental frequency (F0)
- plosive consonants: a **short vertical bright band** on the spectrogram (energy across the whole band, very brief)
- fricatives (such as /s/): sustained broadband energy at high frequencies (4–8 kHz), darker at low frequencies


## Why do we need the mel scale? The secret of human hearing

Earlier we learned the **linear spectrum** (0–8000 Hz, evenly spaced). But that's not how your ears hear.

**Human hearing is logarithmic**, not linear:
- **sensitive at low frequencies**: you can easily tell 100 Hz from 200 Hz (an octave apart — a factor of 2)
- **dull at high frequencies**: you can barely hear the difference between 5000 Hz and 5100 Hz (only 2% apart)

Why? Because your cochlea (inner ear) has its receptors laid out from low to high, and their perception of frequency is on a **logarithmic scale**, not a linear one.

**The definition of the mel scale** imitates exactly this logarithmic behavior. The formula is:

$$\text{Mel}(f) = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

The core idea:
- low frequencies (say 100–500 Hz) get stretched wide on the mel scale (finely divided)
- high frequencies (say 5000–8000 Hz) get squeezed tight on the mel scale (coarse)

In speech recognition we use 40 or 80 **mel bands** (evenly spaced on the mel scale) to cover the whole frequency range. That way:
- more bands subdivide the low frequencies, capturing vowel detail
- a few bands cover the high frequencies, saving computation

That's why a "mel spectrogram" looks like it has "more bright spots at low frequencies" than a linear spectrogram — not because the waveform really has that much more low-frequency energy, but because **we're looking through the lens of human hearing**.

## STFT details: why pad, and how many frames do we get?

When the STFT cuts a signal into frames, the usual question is: **what happens at the edges?**

If the signal has length N, the window length is n_fft and the step is hop, the naive approach is:
```
frame 1: samples [0, n_fft)
frame 2: samples [hop, hop+n_fft)
frame 3: samples [2*hop, 2*hop+n_fft)
...
```

But this **loses the edges**: the last n_fft-hop samples of the signal may not make up a complete frame.

**Aurora's solution**: with `center=True`, reflect-pad `n_fft//2` samples at each end of the signal, so that:
- the center of frame 1 sits exactly on sample 0 (instead of at sample n_fft//2)
- the center of the last frame sits at (or just before) the end of the signal
- so nothing at the edges is lost

**Total length after padding**: $N + n\_fft$ (`n_fft//2` added at each end)

**Counting frames** (this is why `n_frames = 1 + N // hop`):
- after padding, window starts run 0, hop, 2·hop, … up to at most $(N + n\_fft) - n\_fft = N$, so there are $1 + \lfloor N / \text{hop} \rfloor$ frames
- the n_fft added by padding is exactly cancelled by the n_fft each window occupies (the full derivation is in L50, Question 4)

**Checking an example**:
- N = 16000 (1 second @ 16 kHz), hop = 160 (10 ms)
- n_frames = 1 + 16000 // 160 = 1 + 100 = 101 frames
- one frame every 10 ms means about 100 frames per second (plus 1 because of how the formula is defined) — it checks out!

## An FFT detail: why does the output have only n_fft//2 + 1 frequency components?

`np.fft.fft()` returns n_fft complex numbers, one for each of n_fft frequency components. But in speech processing we usually use the **real FFT** (`np.fft.rfft()`), which returns only `n_fft//2 + 1` values. Why?

**The key: the spectral symmetry of real signals**

In the FFT of a real signal, the k-th frequency component and the (n_fft - k)-th are **conjugate-symmetric**:
```
X[k] = conj(X[n_fft - k])
```

That means the real parts are equal and the imaginary parts are opposite. Since the second half is completely determined by the first, storing it is a **waste**. So the real FFT returns only the first half (k = 0 to n_fft//2), `n_fft//2 + 1` values in all:

$$\text{number of bins} = \frac{n\_fft}{2} + 1$$

**What the +1 means**:
- bin 0: the DC component (the signal's mean)
- bins 1, 2, ..., n_fft//2-1: positive frequencies
- bin n_fft//2: the Nyquist frequency (the highest frequency)

So Aurora's `magnitude_spectrogram()` returns shape `(T, n_fft//2+1)`, not `(T, n_fft)`.

In [ ]:
import os

n_fft    = 1024
hop      = n_fft // 4   # 256 samples = 16 ms @ 16kHz
n_mels   = 80
n_mfcc   = 13

mag   = magnitude_spectrogram(samples, n_fft=n_fft, hop_length=hop)  # (T, F)
mel   = mel_spectrogram(samples, sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels)
mel_db = power_to_db(mel)
coeffs = mfcc(samples, sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop, n_mels=n_mels)

n_frames = mag.shape[0]
t_frames = np.arange(n_frames) * hop / sr          # frame-center times (seconds)
t_signal = np.arange(len(samples)) / sr
freqs    = np.linspace(0, sr / 2, mag.shape[1])    # linear frequency axis

fig, axes = plt.subplots(4, 1, figsize=(12, 11), constrained_layout=True)

# ── Waveform ──
ax = axes[0]
ax.plot(t_signal, samples, color="#2196F3", linewidth=0.6)
ax.set_ylabel("amplitude")
ax.set_title("Raw waveform")
ax.set_xlim(t_signal[0], t_signal[-1])
ax.axhline(0, color=plt.rcParams["axes.edgecolor"], linewidth=0.5, linestyle="--")

# ── Magnitude spectrogram (linear Hz, cut at 8 kHz to make speech harmonics easy to see) ──
ax = axes[1]
f_max_idx = np.searchsorted(freqs, min(8000, sr / 2))
im1 = ax.imshow(
    20 * np.log10(mag[:, :f_max_idx].T + 1e-8),
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, freqs[f_max_idx - 1]],
    cmap="magma",
)
plt.colorbar(im1, ax=ax, label="dB")
ax.set_ylabel("frequency (Hz)")
ax.set_title("Magnitude spectrogram (STFT) — harmonic stripes visible in the vowels")

# ── log-Mel ──
ax = axes[2]
im2 = ax.imshow(
    mel_db.T,
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, n_mels],
    cmap="inferno",
)
plt.colorbar(im2, ax=ax, label="dB")
ax.set_ylabel("mel band")
ax.set_title("log-mel spectrogram — perceptual scale, finer at low frequencies")

# ── MFCC ──
ax = axes[3]
im3 = ax.imshow(
    coeffs.T,
    aspect="auto", origin="lower",
    extent=[t_frames[0], t_frames[-1], 0, n_mfcc],
    cmap="RdBu_r",
)
plt.colorbar(im3, ax=ax, label="coefficient value")
ax.set_ylabel("MFCC coefficient")
ax.set_xlabel("time (s)")
ax.set_title("MFCC (13 dims) — C0 ≈ frame energy, C1–C12 describe spectral shape")

plt.suptitle("Aurora audio pipeline: WAV → STFT → Mel → MFCC", fontsize=14)
plt.savefig("L51_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ four-layer visualization done; figure saved to L51_pipeline.png")

print(f"\nShape of each layer:")
print(f"  magnitude_spectrogram : {mag.shape}   (frames, n_fft//2+1)")
print(f"  mel_spectrogram       : {mel.shape}  (frames, n_mels)")
print(f"  mfcc                  : {coeffs.shape}   (frames, n_mfcc)")


## Parameter experiment: the resolution trade-off between n_fft and hop_length

| Parameter | Change | What to expect |
|---|---|---|
| `n_fft` from 512 → 2048 | frequency resolution ↑ (`sr/n_fft`: 31 Hz → 7.8 Hz) | finer harmonic stripes, but worse time resolution (horizontal blur) |
| `hop_length` from 256 → 64 | time resolution ↑ (4× more frames) | plosive consonants get thinner, the matrix gets 4× wider |
| `n_mels` from 40 → 128 | mel-band density ↑ | the log-mel plot gets finer vertically, high formants clearer |

What to watch: with the window length fixed, the two resolutions trade off against each other — a longer window sharpens frequency and blurs time. Speech ASR commonly uses `n_fft=512, hop=160` (a 25 ms window, zero-padded to 512 / a 10 ms step); music analysis often uses a larger `n_fft=2048`.


## What frequency resolution means: why sr / n_fft?

You saw "frequency resolution: sr/n_fft: 31 Hz → 7.8 Hz" in the table. What does it mean?

**Simple explanation**: frequency resolution is the smallest frequency difference the FFT can tell apart. Two adjacent FFT bins (frequency components) are exactly `sr / n_fft` apart.

**Why?**

The k-th FFT bin corresponds to the frequency:
$$f_k = k \cdot \frac{sr}{n\_fft}$$

So two adjacent bins (k and k+1) are separated by:
$$\Delta f = f_{k+1} - f_k = \frac{sr}{n\_fft}$$

**An intuitive example**:
- sr = 16000 Hz, n_fft = 1024
- frequency resolution = 16000 / 1024 ≈ 15.6 Hz
- bin 0 is 0 Hz, bin 1 is 15.6 Hz, bin 2 is 31.2 Hz ...
- if two frequencies are less than 15.6 Hz apart, the FFT puts them in the same bin and can't separate them

**The resolution trade-off**:
- bigger n_fft → better frequency resolution (a more detailed spectrum) → but worse time resolution (each frame covers more samples, so it's less "timely")
- smaller n_fft → better time resolution → but worse frequency resolution (the spectrum "blurs")

The standard choices for speech, n_fft=512/1024, are a compromise between the two.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

configs = [
    dict(n_fft=512,  hop_length=128,  label="n_fft=512, hop=128\n(high time resolution)"),
    dict(n_fft=1024, hop_length=256,  label="n_fft=1024, hop=256\n(balanced, default)"),
    dict(n_fft=2048, hop_length=512,  label="n_fft=2048, hop=512\n(high frequency resolution)"),
]

for ax, cfg in zip(axes, configs):
    label = cfg.pop("label")
    _mag = magnitude_spectrogram(samples, **cfg)
    _freqs = np.linspace(0, sr / 2, _mag.shape[1])
    _f_idx = np.searchsorted(_freqs, 6000)
    _t = np.arange(_mag.shape[0]) * cfg["hop_length"] / sr
    ax.imshow(
        20 * np.log10(_mag[:, :_f_idx].T + 1e-8),
        aspect="auto", origin="lower",
        extent=[_t[0], _t[-1], 0, _freqs[_f_idx - 1]],
        cmap="magma",
    )
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("frequency (Hz)")

plt.suptitle("How n_fft affects spectrogram resolution", fontsize=12)
plt.savefig("L51_nfft_compare.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ parameter comparison figure saved to L51_nfft_compare.png")


## Exercises (TODO)

This section covered the complete WAV → STFT → Mel → MFCC pipeline. The 3 exercises below turn the concepts into hands-on skill.

**Exercise 1**: change the hop and check how the frame count changes

Goal: change `hop_length` to 160 (a 10 ms hop @ 16 kHz), extract the MFCCs again, and check that the frame count ≈ `duration / 0.01`.

**Understanding where the frame-count formula comes from**:
- one frame every 10 ms — how many frames in 1 second? Answer: 100
- more generally, for a signal of `duration` seconds and a hop of `hop/sr` seconds, frames ≈ `duration / (hop/sr)` = `duration * sr / hop`
- with hop=160, sr=16000, frames ≈ `duration * 16000 / 160` = `duration * 100` = `duration / 0.01`
- so the formula `frames ≈ duration / 0.01` is really `duration * sr / hop`, where `sr/hop = 100` (so `hop/sr = 0.01s`)

**Exercise 2**: change n_mels and watch how the MFCCs change

Goal: change `n_mels` from 80 to 40, extract the MFCCs again, check that dimension 0 of the coeffs shape (the frame count) is unchanged, and plot the two sets of MFCCs side by side.

**Hints**:
- the MFCC shape is always `(n_frames, n_mfcc)`; changing n_mels doesn't change the frame count
- but the earlier mel_spectrogram has shape `(n_frames, n_mels)`, so that one does change
- use `plt.subplot()` or `imshow()` to draw the two MFCC heatmaps

**Exercise 3**: run the pipeline with a different fundamental

Goal: implement a `make_vcv_f0` function that raises the fundamental of the vowel /a/ from 120 Hz to 220 Hz, rerun the pipeline, and check on the spectrogram whether the harmonic stripes move up.

**Implementation hints**:
- follow the structure of `make_vcv_signal()`
- for the vowel /a/ part, change `120` to the parameter `f0`
- for the vowel /i/ part, you can keep 150 Hz (or also make it some multiple of f0, say 1.25*f0)
- keep the total duration, the noise segment and the weight vector the same
- check: after MFCC, the new signal's shape should match the original's (same duration)

In [ ]:
# ── Exercise 1: 10 ms hop, check the frame count ──

hop_10ms = 160  # 10 ms @ 16 kHz

# TODO: call mfcc() with hop_10ms to get coeffs_10ms
# Hint: it should look like mfcc(samples, sr, n_mfcc=13, n_fft=1024, hop_length=hop_10ms, n_mels=80)
raise NotImplementedError("TODO: call mfcc() with hop_10ms to get coeffs_10ms")

# Check the frame count is roughly duration / 0.01
# Derivation: duration seconds, hop of hop_10ms/sr seconds, frames ≈ duration / (hop_10ms/sr) = duration * sr / hop_10ms
#      with hop_10ms=160, sr=16000, frames ≈ duration * 100 = duration / 0.01
expected_frames_approx = int(duration / 0.01)
actual_frames = coeffs_10ms.shape[0]
assert abs(actual_frames - expected_frames_approx) <= 5, (
    f"frame count is too far off: expected ≈{expected_frames_approx}, got {actual_frames}. "
    f"Check that hop_length is set to {hop_10ms}; the signal lasts {duration:.3f}s."
)
print(f"✅ Exercise 1 passed: hop=160 (10ms), frames={actual_frames} (expected ≈{expected_frames_approx})")
print(f"   compared with {coeffs.shape[0]} frames at the default hop=256 (16ms) — the frame count is inversely proportional to the hop")

In [ ]:
# ── Exercise 2: n_mels=40 vs n_mels=80, compare the MFCCs ──

# TODO: extract MFCCs with n_mels=40 and with n_mels=80, storing them in coeffs_mel40 and coeffs_mel80
# Hint 1: call mfcc() with the same sr, n_fft, hop_length, n_mfcc, changing only the n_mels argument
# Hint 2: the two results should have exactly the same shape (same n_mfcc); what differs is the number of mel bands in between
raise NotImplementedError("TODO: compute coeffs_mel40 and coeffs_mel80")

# Shape check: same number of frames (T each) and same MFCC dimension (13 each)
assert coeffs_mel40.shape[0] == coeffs_mel80.shape[0], (
    f"frame counts differ: n_mels=40 gives {coeffs_mel40.shape[0]} frames, "
    f"n_mels=80 gives {coeffs_mel80.shape[0]} frames. Is hop_length the same?"
)
assert coeffs_mel40.shape[1] == coeffs_mel80.shape[1] == 13, (
    "the n_mfcc dimension should be 13 for both"
)
print(f"✅ Exercise 2 passed:")
print(f"   n_mels=40  → shape={coeffs_mel40.shape}   (frames, 13 coefficients)")
print(f"   n_mels=80  → shape={coeffs_mel80.shape}  (frames, 13 coefficients)")
print(f"   same frame count and coefficient count: n_mels doesn't affect the MFCC output dimensions (fixed at 13 in the Mel→DCT stage)")

# Optional, going further: plot the two MFCCs side by side (see the plotting code in "3. Visual check")
# e.g. draw two heatmaps with plt.imshow(), or compute the Euclidean distance to see how close they are

In [ ]:
# ── Exercise 3: change F0, watch the harmonics move up ──

def make_vcv_f0(f0: float = 220, sr: int = 16000) -> np.ndarray:
    """Synthesize a V-C-V with the same structure as make_vcv_signal, but with an adjustable fundamental for the vowel /a/.

    TODO: implement this function

    Hints:
    1. Follow the structure of make_vcv_signal(), in three parts: vowel /a/, plosive, vowel /i/
    2. For the vowel /a/ part: change `120` to f0
    3. For the vowel /i/ part: keep 150 Hz, or change it to 1.25*f0 (either is fine)
    4. Keep everything else (durations, noise, harmonic weights) the same
    5. Make sure the returned signal has the same length as make_vcv_signal() (same time structure)
    """
    raise NotImplementedError("Exercise 3: implement make_vcv_f0")

signal_f0_220 = make_vcv_f0(f0=220, sr=sr)

# Check: extract the MFCCs; the shape should match the original coeffs (same duration, same hop)
coeffs_f0_220 = mfcc(signal_f0_220, sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80)
assert coeffs_f0_220.shape == coeffs.shape, (
    f"shape mismatch: f0=220 gives {coeffs_f0_220.shape}, expected {coeffs.shape}. "
    "Make sure make_vcv_f0 produces a signal as long as make_vcv_signal's."
)
print(f"✅ Exercise 3 passed: F0=220 Hz MFCC shape={coeffs_f0_220.shape}, same as the original")
print("  What to look for:")
print("  - rerun the magnitude-spectrogram code from '3. Visual check' on the new signal")
print("  - you'll see the harmonic stripes go from ~120 Hz spacing (stripes 1, 2, 3... from the bottom) to ~220 Hz spacing")
print("  - which confirms it: changing F0 (the fundamental) directly changes the harmonic structure")

## Wrap-up

This section loaded a WAV file with `read_wav` (float64, [-1,1]) and then called `magnitude_spectrogram`, `mel_spectrogram` and `mfcc` in turn, with output shapes `(T, n_fft//2+1)`, `(T, n_mels)` and `(T, n_mfcc)`.
The plots make the vowels' harmonic stripes and the consonant's broadband transient visible at a glance; the optional librosa is only there as an answer key, helping confirm that Aurora's Audio Core is numerically correct.
Every calculation comes from `aurora.audio` (`io.py / stft.py / mel.py / mfcc.py`), with zero dependency on librosa or scipy.signal, in line with the Audio Core principle in CLAUDE.md.
The next lesson (L52) rounds this chain off as the Audio Core completion check: all 39 unit tests green, assembling the evidence of what you've truly mastered and can explain, and confirming the MFCC pipeline's end-to-end output is correct.


## DCT and MFCC: why is C0 the total energy?

The last step from mel spectrum to MFCC is the **discrete cosine transform (DCT)**. You might wonder: why transform yet again?

**What the DCT does**: adjacent mel bands are strongly **correlated** (neighboring frequencies usually have similar energy). The DCT turns that correlation into a **sparse representation** — most of the energy concentrated in a few coefficients. That's great for data compression and for training neural networks.

**The DCT's mathematical definition** (DCT-II, the most common):

For an input vector of length N, $x = [x_0, x_1, ..., x_{N-1}]$, the DCT's k-th output coefficient is:

$$C_k = \sum_{n=0}^{N-1} x_n \cos\left(\frac{\pi k (2n+1)}{2N}\right)$$

The 0th coefficient is:

$$C_0 = \sum_{n=0}^{N-1} x_n \cos(0) = \sum_{n=0}^{N-1} x_n$$

which is **the sum of all the inputs**! After normalization (dividing by $\sqrt{N}$) it becomes:

$$C_0 = \frac{1}{\sqrt{N}} \sum_{n=0}^{N-1} x_n$$

**Intuitively**: when k=0, the cosine becomes the constant 1, so the DCT turns into a sum.

**What it means in MFCC**:
- input x = log-mel energies (40 or 80 dims)
- C0 = the sum of the log-mel energies ≈ **the frame's overall loudness**
- C1, C2, ... = the **shape** of the mel energies (the contour of the spectrum)

**Why is C0 often dropped?**
- in speech recognition we care about **what** is said (the phonemes), not **how loudly**
- C0 mainly encodes loudness, and loudness has little to do with the phoneme class
- dropping C0 also reduces the feature dimension and speeds up computation

**Checking an example**:

In [ ]:
# Check: the DCT's 0th coefficient = sum of the inputs / √N
from aurora.audio.mfcc import dct_ii

# Simple example: 3 numbers
x_simple = np.array([1.0, 2.0, 1.0])
c_simple = dct_ii(x_simple)
expected_c0 = np.sum(x_simple) / np.sqrt(len(x_simple))
print(f"input: {x_simple}")
print(f"DCT output: {c_simple}")
print(f"C0 = {c_simple[0]:.6f}")
print(f"expected C0 = sum(x)/√N = {expected_c0:.6f}")
assert abs(c_simple[0] - expected_c0) < 1e-10, "check failed!"
print("✅ the DCT's 0th coefficient really is sum(x)/√N\n")

# Realistic example: mel energies all equal to 1 (simulating uniform energy)
x_ones = np.ones(40)
c = dct_ii(x_ones)
expected_c0_40 = np.sum(x_ones) / np.sqrt(40)
print(f"an all-ones vector of length 40 through the DCT:")
print(f"C0 = {c[0]:.6f}, expected = {expected_c0_40:.6f}")
assert abs(c[0] - expected_c0_40) < 1e-10
print("✅ check passed")
print(f"\nThis shows: if the mel energy is spread evenly over 40 bands, C0 is a normalized value of the total energy")

## ✏️ Whiteboard challenge: MFCC engineering parameters by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: `sr=16000, n_fft=1024, hop=256, duration=1.0s`:
- total samples N = ?
- aurora's `stft` defaults to `center=True` (reflect-padding `n_fft//2` at each end), so n_frames = 1 + N // hop = ?
- MFCC matrix shape = ? (n_frames × n_mfcc=13)

**Question 2**: `hop=160` (10 ms @16kHz) vs `hop=256` (16 ms):
- how many times as many frames does the 10 ms hop give as the 16 ms hop?
- time resolution goes up — but what happens to the amount of computation?

**Question 3**: the harmonic structure of a 440 Hz fundamental:
- at which frequencies do the harmonics appear? (f, 2f, 3f, ... up to sr/2=8000 Hz)
- with `n_fft=1024, sr=16000`, which FFT bin does 440 Hz land in?

**Question 4**: why is the 0th MFCC coefficient (C0) often dropped?
(Hint: C0 ∝ sum(log mel energy) ∝ overall loudness, which has nothing to do with recognizing the phoneme)

**Question 5**: given two MFCC sequences, how would you judge the phonetic "distance" between them?
(Hint: DTW, dynamic time warping, or a simple Euclidean distance)

Once you've worked it out, run the cell below to check your answers.

## Why do the harmonics stop at 8000 Hz? The Nyquist limit

In the whiteboard questions and the code you'll often see the limit "harmonic frequencies up to sr/2". Why?

Remember the sampling theorem? A system with a 16000 Hz sample rate can capture signals only up to 8000 Hz (sr/2). Any frequency component above 8000 Hz gets **aliased** into some other frequency — not what we want.

So when generating synthetic speech or analyzing harmonics, we need to make sure:
- none of the harmonics of the fundamental f0 go above 8000 Hz
- the harmonic range = f0, 2×f0, 3×f0, ..., up to the last harmonic ≤ 8000 Hz

For example:
- f0 = 440 Hz: 8000 / 440 ≈ 18.2, so there are 18 harmonics (k=1 to 18)
- f0 = 100 Hz: 8000 / 100 = 80, so there are 80 harmonics

This ensures the signal we capture isn't distorted.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: parameter calculation (aurora stft defaults to center=True: reflect-pad n_fft//2 at each end → n_frames = 1 + N//hop)
sr, n_fft, hop, dur = 16000, 1024, 256, 1.0
N = int(sr * dur)
n_frames = 1 + N // hop
n_mfcc = 13
assert N == 16000
assert n_frames == 63
print(f"Q1 ✅  N={N}, n_frames={n_frames}, MFCC shape=({n_frames},{n_mfcc})")

# Q2: frame-count comparison
hop1, hop2 = 160, 256  # 10ms vs 16ms
n_frames1 = 1 + N // hop1
n_frames2 = 1 + N // hop2
ratio = n_frames1 / n_frames2
assert ratio > 1.5
print(f"Q2 ✅  10ms: {n_frames1} frames vs 16ms: {n_frames2} frames, frame ratio={ratio:.2f} (computation grows in proportion)")

# Q3: harmonics and bins
f0 = 440
harmonics = [f0 * k for k in range(1, int(8000/f0)+1)]
bins = [round(f * n_fft / sr) for f in harmonics]
assert bins[0] == round(440 * 1024 / 16000)
print(f"Q3 ✅  440 Hz harmonics: {harmonics[:5]}... Hz, bins: {bins[:5]}...")

# Q4: C0 = overall loudness (unrelated to the phoneme)
# Check: DCT of an all-ones energy vector, C0=sum
from aurora.audio.mfcc import dct_ii
x_ones = np.ones(40)
c = dct_ii(x_ones)
assert abs(c[0] - np.sum(x_ones) / np.sqrt(40)) < 1e-10
print(f"Q4 ✅  C0={c[0]:.4f}=sum(x)/√N={np.sum(x_ones)/np.sqrt(40):.4f} (total energy, unrelated to the phoneme)")

# Q5: proof of concept for MFCC distance (Euclidean distance)
mfcc_a = np.array([1.0, 0.5, -0.3] + [0]*10)
mfcc_b = np.array([1.1, 0.4, -0.25] + [0]*10)
dist = np.linalg.norm(mfcc_a - mfcc_b)
assert dist > 0 and dist < 1.0
print(f"Q5 ✅  Euclidean distance between two MFCC frames={dist:.4f} (small → similar phonemes, large → different phonemes)")
print("\n🎉 MFCC engineering whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l51_review = {
    "wav_pipeline_ran":        None,  # ran the full WAV→MFCC pipeline and saw the 4-layer visualization? True/False
    "exercise1_hop_done":      None,  # Exercise 1 done: frame counts for a 10ms vs 16ms hop? True/False
    "exercise2_mels_done":     None,  # Exercise 2 done: MFCCs for n_mels=40 vs 80? True/False
    "harmonic_structure":      None,  # understand vowel = harmonic structure, F0 = fundamental? True/False
    "whiteboard_passed":       None,  # worked through the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l51_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l51_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L51 fully cleared! On to L52: Audio Core complete')

---

→ **Next lesson** · [L52 · Audio Core complete](L52_features_done.ipynb)

> Next lesson covers **Audio Core complete**: wrapping up feature engineering, all 39 unit tests green, and assembling the evidence of what you truly understand and can explain.